# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YassinKhodir/yassin-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is a **ranking/scoring task**. The system will assign each eligible content page a continuous review-priority score, then order pages from highest to lowest priority. The score is decision support for an SEO or content editor; it is not an automatic refresh decision.

In [1]:
task_frame = {
    "task_type": "ranking/scoring",
    "decision": "which pages an editor reviews first",
    "output": "continuous review-priority score and ranked queue",
}
task_frame

{'task_type': 'ranking/scoring', 'decision': 'which pages an editor reviews first', 'output': 'continuous review-priority score and ranked queue'}

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter exercise, the provisional proxy is `is_declining_label`, which equals 1 when the current-window `trend_direction` is `down`. This is a **defined teaching proxy**, not an ideal real-world target. Because `trend_direction` comes from `trend_pct`, both must be excluded from the features. For the stronger warehouse project, the target will be an observed future outcome: whether a page's search impressions decline in a later 30-day window using only features measured before that window. The model's probability of that future outcome becomes the review-priority score.

In [2]:
from pathlib import Path
import pandas as pd

data_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((path for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Run this notebook from the repository root or work/notebooks directory.")

df = pd.read_csv(data_path)
df["is_declining_proxy"] = df["trend_direction"].eq("down").astype(int)
proxy_summary = {
    "proxy_source": "trend_direction == 'down'",
    "positive_rows": int(df["is_declining_proxy"].sum()),
    "positive_rate": round(df["is_declining_proxy"].mean(), 4),
    "excluded_from_features": ["trend_direction", "trend_pct"],
}
proxy_summary

{'proxy_source': "trend_direction == 'down'", 'positive_rows': 16262, 'positive_rate': np.float64(0.5421), 'excluded_from_features': ['trend_direction', 'trend_pct']}

## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary metric is **precision@K**, where K is the number of pages the editor can review. It measures the share of the top-K recommendations that later show the observed outcome. A useful first threshold is at least **20% relative lift over the base rate on held-out clients**. The threshold must be fixed before model comparison; it cannot be chosen after seeing the final test result.

In [3]:
base_rate = df["is_declining_proxy"].mean()
minimum_relative_lift = 0.20
minimum_precision_at_k = min(1.0, base_rate * (1 + minimum_relative_lift))
print(f"Starter proxy base rate: {base_rate:.1%}")
print(f"Pre-registered success threshold: precision@K >= {minimum_precision_at_k:.1%}")

Starter proxy base rate: 54.2%
Pre-registered success threshold: precision@K >= 65.0%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents **one pseudonymized content page**. `content_id` and `client_id` are context fields used for uniqueness checks, grouping, and client-held-out validation; they are not model features. The dataframe below shows the grain and a few observable fields. CTR is stored as percentage points, so `0.76` means 0.76%, and `avg_position = 0` means missing position data.

In [4]:
assert len(df) == df["content_id"].nunique(), "content_id must be unique at the page grain"
print(f"Grain check passed: {len(df):,} rows and {df['content_id'].nunique():,} unique content pages.")

display_columns = [
    "content_id", "client_id", "impressions_90d", "ctr",
    "avg_position", "days_since_last_update", "trend_direction",
]
df[display_columns].head()

Grain check passed: 30,000 rows and 30,000 unique content pages.


             content_id  ... trend_direction
0  content_304f48230142  ...            down
1  content_a1fb4e703a9e  ...            down
2  content_9aa793d4d895  ...            down
3  content_331d6c4de07b  ...          stable
4  content_d99b7a2d90ca  ...            down

[5 rows x 7 columns]

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as “review every page with low CTR” ignores volume, position, content age, freshness, and engagement. Those signals interact, and the same CTR can mean something different for a page with 20 impressions than for one with 20,000. A simple rule remains necessary as a transparent baseline, but an honestly validated model may combine the signals into a more useful ordering. Humans still make the final content decision.

In [5]:
candidate_signal_groups = {
    "visibility": ["impressions_90d", "clicks_90d", "avg_position", "ctr"],
    "engagement": ["sessions_90d", "engagement_rate", "scroll_rate"],
    "freshness": ["content_age_days", "days_since_last_update"],
}
leakage_exclusions = ["trend_direction", "trend_pct"]
{"candidate_signal_groups": candidate_signal_groups, "leakage_exclusions": leakage_exclusions}

{'candidate_signal_groups': {'visibility': ['impressions_90d', 'clicks_90d', 'avg_position', 'ctr'], 'engagement': ['sessions_90d', 'engagement_rate', 'scroll_rate'], 'freshness': ['content_age_days', 'days_since_last_update']}, 'leakage_exclusions': ['trend_direction', 'trend_pct']}

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.